[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/erwincarlogonzales/mldetection-YOLO/blob/main/YOLO_Detection_Counting_MLflow_Experiments_YOLOv8n.ipynb)

# GITHUB SETUP
link: https://github.com/erwincarlogonzales

In [ ]:
# Github
from google.colab import userdata

GIT_TOKEN = userdata.get('GITHUB_TOKEN')
GIT_USERNAME = 'erwincarlogonzales'
GIT_REPO = 'yolo-object-counter-mlflow'
CLONE_URL = f"https://{GIT_TOKEN}@github.com/{GIT_USERNAME}/{GIT_REPO}.git"

# Clone repo
!git clone '{CLONE_URL}'

In [ ]:
# Go to current dir in github
%cd {GIT_REPO}

# dir
!ls -la

In [ ]:
# Configure Your Git ID
!git config --global user.name "erwincarlogonzales"
!git config --global user.email "erwincarlogonzales@gmail.com"

# GIT COMMIT
- Do not run unless you are going to commit and push

**How to save on GitHub**
1. Go to File > Download.ipynb
2. Go to Folder Icon on the Left and select yolo-object-counter-mlflow folder
3. Click the 3 dots on the right and select Upload
4. Go to where you downloaded the .ipynb and select that
5. This will overwrite the current notebook but you wont see that happen
6. Proceed to !git status to commit and push

In [ ]:
!git status

In [ ]:
!git add .
!git commit -m 'training yolo8n model'

In [ ]:
!git push origin main

# SETUP & DATASET PREPARATION

In [ ]:
# Install Ultralytics & Roboflow
!pip install ultralytics roboflow mlflow onnx onnxruntime -q

In [ ]:
# Download Dataset from Roboflow
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("objectdetection-fvcmc").project("hardware-object-detection-xw2gx")
version = project.version(5)
dataset = version.download("yolov8")

In [ ]:
# Store the path to data.yaml for training
DATASET_YAML = f"{dataset.location}/data.yaml"

# Check of the data.yaml path
print(f"Dataset YAML path: {DATASET_YAML}")
!cat {DATASET_YAML}

# MLFLOW AND MODEL CONFIGURATION

In [ ]:
# User defined inputs
import mlflow
import os

MLFLOW_TRACKING_URI = 'https://dagshub.com/erwincarlogonzales/mldetection-YOLO.mlflow'
EXPERIMENT_NAME = 'yolo8n_training'

# Get credentials and set environment variables
os.environ.update({
    'MLFLOW_TRACKING_URI': MLFLOW_TRACKING_URI,
    'MLFLOW_TRACKING_USERNAME': userdata.get('MLFLOW_TRACKING_USERNAME'),
    'MLFLOW_TRACKING_PASSWORD': userdata.get('MLFLOW_TRACKING_PASSWORD')
})

In [ ]:
# Check MLflow connection
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Successfully connected to MLflow {MLFLOW_TRACKING_URI}")
print(f"Using experiment: {EXPERIMENT_NAME}")

In [ ]:
# Determine device for training
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

In [ ]:
# Model Config
from ultralytics import YOLO

CONFIG = {
    'model_variant_source': 'yolov8n.pt',
    'epochs': 50,
    'image_size': 640,
    'batch_size': 16,
    'patience': 3,
    'ultralytics_project_folder': "yolo8_colab_runs",
    'device': device,
    'dataset_yaml_path': DATASET_YAML,
    'ops_set': 13
}

# TRAINING AND MLFLOW EXPORT

In [ ]:
from ultralytics.utils import SETTINGS
from onnxruntime.quantization import quantize_dynamic, QuantType

def train_export_log_yolo_bare(config):

    SETTINGS.update({'mlflow': False})

    run_name = f"{config['model_variant_source'].split('.')[0]}_Ep{config['epochs']}_Bs{config['batch_size']}_Img{config['image_size']}"
    local_run_output_dir = None
    model_name_stem = "best" # Using "best" for consistent naming of exported files

    with mlflow.start_run(run_name=run_name) as run:
        original_run_id = run.info.run_id
        print(f"MLflow Run ID started: {original_run_id} (Name: {run_name})")

        mlflow.log_params({
            'model_variant': config['model_variant_source'],
            'epochs': config['epochs'],
            'image_size': config['image_size'],
            'batch_size': config['batch_size'],
            'patience': config['patience'],
            'dataset_yaml': os.path.basename(config['dataset_yaml_path']),
            'device_used': config['device']
        })

        model = YOLO(config['model_variant_source'])
        model.to(config['device'])

        print(f"\nStarting YOLOv8 training for {config['epochs']} epochs...")
        results = model.train(
            data=config['dataset_yaml_path'],
            epochs=config['epochs'],
            imgsz=config['image_size'],
            batch=config['batch_size'],
            patience=config['patience'],
            project=config['ultralytics_project_folder'],
            name=original_run_id,
            device=config['device'],
            exist_ok=True
        )
        print("Training complete.")

        # Get local output dir
        run_output_dir = results.save_dir
        local_run_output_dir = run_output_dir
        print(f"YOLO outputs saved locally to: {run_output_dir}")

        # Log metrics on MLflow
        print("\nLogging metrics to MLflow...")
        mlflow.log_metric("mAP_0.5_0.95", results.box.map)
        mlflow.log_metric("mAP_0.5", results.box.map50)
        mlflow.log_metric("Precision", results.box.mp)
        mlflow.log_metric("Recall", results.box.mr)
        print(f"  Metrics logged (mAP@0.5-0.95: {results.box.map:.4f})")

        artifacts_to_log = ["results.csv",
                            "confusion_matrix.png",
                            "PR_curve.png",
                            "F1_curve.png",
                            "P_curve.png",
                            "R_curve.png",
                            "labels.jpg",
                            "labels_correlogram.jpg",
                            "val_batch0_labels.jpg",
                            "val_batch0_pred.jpg",
                            "results.png"
                            ]

        print("Logging training artifacts to MLflow under 'training_artifacts'...")
        for artifact_name in artifacts_to_log:
            artifact_path = os.path.join(run_output_dir, artifact_name)
            if os.path.exists(artifact_path):
                 mlflow.log_artifact(artifact_path, artifact_path="training_artifacts")
            else:
                 print(f"  Skipping non-critical training artifact (not found): {artifact_name}")

        print("\nStarting model export and logging to MLflow...")
        best_model_path_pt = os.path.join(run_output_dir, 'weights/best.pt')

        print(f"Logging PyTorch model: {best_model_path_pt}")
        mlflow.log_artifact(best_model_path_pt, artifact_path="models/pytorch")
        print(f"  Logged PyTorch model: {model_name_stem}.pt")

        export_model = YOLO(best_model_path_pt)

        # 1. ONNX FP32 Export & Log
        print("\nExporting and logging ONNX FP32...")
        fp32_onnx_path = export_model.export(
            format='onnx',
            imgsz=config['image_size'],
            simplify=True,
            device=config['device']
        )
        mlflow.log_artifact(fp32_onnx_path, artifact_path="models/onnx_fp32")
        print(f"  Logged ONNX FP32: {os.path.basename(fp32_onnx_path)}")

        # 2. ONNX INT8 Dynamic Quantization & Log
        print("\nPerforming ONNX INT8 Dynamic Quantization...")
        quantized_model_dir = os.path.dirname(fp32_onnx_path)
        output_onnx_int8_dynamic_filename = f"{model_name_stem}_int8_dynamic.onnx"
        output_onnx_int8_dynamic_path = os.path.join(quantized_model_dir, output_onnx_int8_dynamic_filename)

        quantize_dynamic(
            model_input=fp32_onnx_path,
            model_output=output_onnx_int8_dynamic_path,
            weight_type=QuantType.QUInt8
        )
        print(f"  Dynamically quantized ONNX model (INT8) saved to: {output_onnx_int8_dynamic_path}")

        mlflow.log_artifact(output_onnx_int8_dynamic_path, artifact_path="models/onnx_int8_dynamic")
        print(f"  Logged ONNX INT8 Dynamic: {os.path.basename(output_onnx_int8_dynamic_path)}")

        # 3. ONNX FP16 Export & Log
        print("\nExporting and logging ONNX FP16...")
        fp16_onnx_path = export_model.export(
            format='onnx',
            imgsz=config['image_size'],
            half=True,
            simplify=True,
            device=config['device']
        )
        mlflow.log_artifact(fp16_onnx_path, artifact_path="models/onnx_fp16")
        print(f"  Logged ONNX FP16: {os.path.basename(fp16_onnx_path)}")

        # 4. TensorRT FP16
        print("\nExporting and logging TensorRT FP16 Engine...")
        fp16_engine_path = export_model.export(
            format='engine',
            imgsz=config['image_size'],
            half=True,
            device=config['device']
        )
        mlflow.log_artifact(fp16_engine_path, artifact_path="models/tensorrt_fp16")
        print(f"  Logged TensorRT FP16: {os.path.basename(fp16_engine_path)}")

        # 5. TensorRT INT8 -> Your session crashed. Automatically restarting when converting
        print("\nExporting and logging TensorRT INT8 Engine...")
        int8_engine_path = export_model.export(
            format='engine',
            imgsz=config['image_size'],
            int8=True,
            data=config['dataset_yaml_path'],
            device=config['device'],
            batch=config['batch_size']
        )
        mlflow.log_artifact(int8_engine_path, artifact_path="models/tensorrt_int8")
        print(f"  Logged TensorRT INT8: {os.path.basename(int8_engine_path)}")

        print("\nMLflow Run operations complete.")

    return local_run_output_dir

# Run training and return output dir
output_directory = train_export_log_yolo_bare(CONFIG)
if output_directory:
    print(f"\nTraining run local output directory: {output_directory}")

In [ ]:
from IPython.display import Image, display

# Display confusion
confusion_matrix_path = os.path.join(output_directory, 'confusion_matrix.png')
display(Image(filename=confusion_matrix_path, width=600))

In [ ]:
from IPython.display import Image, display

# Display training results
results_path = os.path.join(output_directory, 'results.png')
display(Image(filename=results_path, width=600))

In [ ]:
from IPython.display import Image, display

# Diplay validation
validation_image_file = os.path.join(output_directory, 'val_batch0_pred.jpg')
display(Image(filename=validation_image_file, width=600))

# TESTING MODELS

### TensorRT

In [ ]:
# Load the exported TensorRT INT8 model
engine_model = YOLO("/content/yolo-object-counter-mlflow/yolo8_colab_runs/d60c7b018ce64ccb85bedcfb478c70fb/weights/best.engine", task="detect")

# Run inference
result = engine_model.predict("/content/yolo-object-counter-mlflow/hardware-object-detection-5/test/images/black-combo_frame1055_jpg.rf.b8b6f7202f139d3cf5a0c738d1a822e8.jpg")

### ONNX

In [ ]:
# Load the exported ONNX INT8 dynamic model
onnx_model  = YOLO("/content/yolo-object-counter-mlflow/yolo8_colab_runs/d60c7b018ce64ccb85bedcfb478c70fb/weights/best_int8_dynamic.onnx", task="detect")

# Run inference
result = onnx_model.predict("/content/yolo-object-counter-mlflow/hardware-object-detection-5/test/images/tek-screw-combo_frame950_jpg.rf.ceb2f9cbf9aec4250b968337717caf69.jpg")

# REGISTER MODEL

In [ ]:
from mlflow.tracking import MlflowClient

# Model registry
def register_model(run_id, model_name):
    model_uri = f'runs:/{run_id}/model'

    return mlflow.register_model(model_uri=model_uri, name=model_name)

# Promote model
def promote_challenger_to_production(model_name, prod_name):
    client = MlflowClient()
    current_model_uri = f"models:/{model_name}@challenger"
    client.copy_model_version(src_model_uri=current_model_uri, dst_name=prod_name)

In [ ]:
run_id = 'd60c7b018ce64ccb85bedcfb478c70fb' # Get this from MLflow UI
model_name = 'best_int8_dynamic.onnx'
prod_name = 'yolo8n_android_production'

# Register model
model_details = register_model(run_id, model_name)
print(f'Registered model version: {model_details.version}')

In [ ]:
# Promote challenger to production => make sure to add challenger to the model alias
promote_challenger_to_production(model_name, prod_name)